# Measurement Profiles

A bar of the cycle progression barcode is not carried by one fixed cycle: its
cycle representative changes as the filtration grows. A *measurement profile*
records how a scalar measurement of that representative, such as its length or
enclosed area, evolves along the bar.

For a `cycle_func(chain, point_cloud) -> float`, the profile of a bar is a step
function on the half-open bar interval `[birth, death)`: each representative is
active on its own half-open interval `[active_start, active_end)` and contributes
the value of `cycle_func` there.

This notebook computes profiles, inspects their values, plots them next to the
barcode, and animates them along the filtration. The next tutorial,
`05_measurement_landscapes.ipynb`, summarizes profiles into landscapes and
feature vectors.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import HTML
from persforest import PersistenceForest
from persforest.measurements.basic import signed_chain_area, signed_chain_edge_length
from persforest.measurements.shape import signed_chain_circularity


def sample_noisy_star(rng, points_per_edge=14, inner_radius=0.48, outer_radius=1.2, noise=0.02):
    angles = np.linspace(np.pi / 2, np.pi / 2 + 2.0 * np.pi, 10, endpoint=False)
    radii = np.where(np.arange(10) % 2 == 0, outer_radius, inner_radius)
    vertices = np.column_stack((radii * np.cos(angles), radii * np.sin(angles)))
    t = np.linspace(0.0, 1.0, points_per_edge, endpoint=False)[:, None]
    points = np.vstack([start + t * (end - start) for start, end in zip(vertices, np.roll(vertices, -1, axis=0))])
    return points + rng.normal(scale=noise, size=points.shape)


def sample_noisy_circle(rng, center, radius, n_points, noise=0.02):
    theta = np.linspace(0.0, 2.0 * np.pi, n_points, endpoint=False)
    points = np.asarray(center) + radius * np.column_stack((np.cos(theta), np.sin(theta)))
    return points + rng.normal(scale=noise, size=points.shape)

## Build an example with several loops

A star, a circle and a smaller circle give three loops of different size and
shape. Later in the filtration, a fourth loop forms in the gap between the shapes.

In [ ]:
rng = np.random.default_rng(11)
points = np.vstack((
    sample_noisy_star(rng),
    sample_noisy_circle(rng, center=(2.3, 0.4), radius=0.55, n_points=40),
    sample_noisy_circle(rng, center=(2.1, -1.0), radius=0.3, n_points=24),
))
forest = PersistenceForest(points)

MIN_BAR_LENGTH = 0.05

fig, axes = plt.subplots(1, 2, figsize=(10, 3.8))
axes[0].scatter(points[:, 0], points[:, 1], s=6, color="black")
axes[0].set_aspect("equal")
axes[0].set_title("Point cloud")
forest.plot_barcode(ax=axes[1], min_bar_length=MIN_BAR_LENGTH, coloring="forest")
plt.show()

## Compute profiles

`compute_measurement_profiles` evaluates `cycle_func` once on every representative
of every bar at least `min_bar_length` long. With `cache=True` (the default) the
result is also stored in `forest.measurement_profiles[label]`.

By default, opposite-oriented duplicate simplices are cancelled before measuring.
Pass `signed=True` to measure the stored signed chains instead.

In [ ]:
area = forest.compute_measurement_profiles(signed_chain_area, "area", min_bar_length=MIN_BAR_LENGTH)

[(bar.bar_id, round(bar.birth, 3), round(bar.death, 3)) for bar in area.bars]

Indexing the result with a bar of the forest (or a captured bar id) returns its
step function. `starts`, `ends` and `vals` hold the half-open pieces;
`eval_on_grid` samples it, with the `baseline` (0 by default) outside the bar.

In [ ]:
longest = forest.max_bar()
step = area[longest]
print(f"bar [{longest.birth:.3f}, {longest.death:.3f}) has {len(step.vals)} pieces")
for start, end, value in list(zip(step.starts, step.ends, step.vals))[:5]:
    print(f"  [{start:.3f}, {end:.3f}): area {value:.3f}")

grid = np.linspace(0.0, 0.8, 9)
np.round(area.evaluate_on_grid(grid), 3)  # one row per bar

## Plot profiles next to the barcode

`plot_measurement_profiles` overlays the profiles of all bars. It selects bars
like `plot_barcode` (`min_bar_length`, `max_bars`), uses the same `coloring`, and
the same filtration-axis limits, so the panels line up. The measurement axis
starts at 0.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.2), gridspec_kw={"width_ratios": [1.1, 1, 1.2]})
forest.plot_barcode_cycle_reps(min_bar_length=MIN_BAR_LENGTH, relative_position=0.3, ax=axes[0],
                               show=False, title="representatives at 30% of each bar")
forest.plot_barcode(ax=axes[1], min_bar_length=MIN_BAR_LENGTH, coloring="forest", bar_width=4)
forest.plot_measurement_profiles(area, ax=axes[2], min_bar_length=MIN_BAR_LENGTH, legend=True)
fig.tight_layout()
plt.show()

## Select bars and style curves

- `max_bars` keeps the longest bars, `bars` restricts the plot to given bars.
- `coloring` takes the barcode schemes `"forest"`, `"bars"`, `"none"` and `"grey"`.
- `extend_baseline=True` draws each profile over the whole filtration axis,
  constant at 0 outside its bar (off by default).
- Other keywords such as `linewidth` or `color` are passed to every curve.

Profiles can also be selected by their cached label.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
forest.plot_measurement_profiles("area", ax=axes[0], max_bars=2, coloring="bars", linewidth=2,
                                 legend=True, title="two longest bars")
forest.plot_measurement_profiles("area", ax=axes[1], bars=[longest], color="black",
                                 linewidth=2, extend_baseline=True,
                                 title="longest bar, zero outside its lifespan")
fig.tight_layout()
plt.show()

## Compare measurements

Different measurements tell different stories about the same bars. Edge length
and area shrink as the representatives tighten. Circularity stays close to 1 for
the two round loops, while the star starts low and becomes rounder as its
representative cuts off the spikes.

In [ ]:
measurements = [
    ("edge length", signed_chain_edge_length),
    ("area", signed_chain_area),
    ("circularity", signed_chain_circularity),
]

fig, axes = plt.subplots(1, len(measurements), figsize=(15, 3.8))
for ax, (label, cycle_func) in zip(axes, measurements):
    profiles = forest.compute_measurement_profiles(cycle_func, label, min_bar_length=MIN_BAR_LENGTH)
    forest.plot_measurement_profiles(profiles, ax=ax, linewidth=1.8)
fig.tight_layout()
plt.show()

## Animate profiles along the filtration

`animate_barcode_measurement` shows the filtration next to the profiles. A
vertical line tracks the current filtration value and a dot marks the current
value of every bar alive at that moment. The profile panel uses the bars and
colors of the filtration panel; pass `bar=` to follow a single bar, or
`measurement_kwargs` to change the profile plot.

In [ ]:
anim, anim_fig = forest.animate_barcode_measurement(
    signed_chain_area,
    signed=False,
    frames=30,
    t_max=0.8,
    total_figsize=(11, 4),
    filtration_kwargs={"vertex_size": 4, "min_bar_length": MIN_BAR_LENGTH},
    measurement_kwargs={"ylabel": "area", "title": "area profiles"},
)
plt.close(anim_fig)
HTML(anim.to_jshtml())

## Next steps

Profiles of different bars live on different intervals. Measurement landscapes,
introduced in `05_measurement_landscapes.ipynb`, rank the profiles at every
filtration value and give fixed-length feature vectors that can be compared
across forests.